In [ ]:
# Paso 1: Importar librerías
import pandas as pd

# Paso 2: Cargar archivos
dim_product = pd.read_excel("DIM_PRODUCT (1).xlsx")
dim_segment = pd.read_excel("DIM_SEGMENT (1).xlsx")
dim_calendar = pd.read_excel("DIM_CALENDAR (2).xlsx")
fact_sales = pd.read_csv("FACT_SALES (1).csv")
print("Archivos cargados correctamente")

# Paso 3: Revisar datos
for df, name in [(dim_product,"DIM_PRODUCT"), (dim_segment,"DIM_SEGMENT"),
                 (dim_calendar,"DIM_CALENDAR"), (fact_sales,"FACT_SALES")]:
    print(f"\n{name}")
    print(df.head())          # primeras filas
    print(df.info())          # estructura y tipos de datos
    print(df.describe(include="all"))  # estadísticas generales

# Paso 4: Limpieza de datos
def limpiar(df):
    df = df.drop_duplicates()          # elimina duplicados
    df = df.fillna("NO DEFINIDO")      # rellena nulos
    return df

dim_product = limpiar(dim_product)
dim_segment = limpiar(dim_segment)
dim_calendar = limpiar(dim_calendar)
fact_sales = limpiar(fact_sales)
print("Datos limpios")

# Paso 5: Unir tablas
# Une FACT_SALES con DIM_PRODUCT
df_merge = pd.merge(fact_sales, dim_product,
                    left_on="ITEM_CODE", right_on="ITEM", how="left")

# Une con DIM_SEGMENT usando las columnas comunes
df_merge = pd.merge(df_merge, dim_segment,
                    on=["CATEGORY","ATTR1","ATTR2","ATTR3","FORMAT"], how="left")

# Une con DIM_CALENDAR por la columna WEEK
df_merge = pd.merge(df_merge, dim_calendar, on="WEEK", how="left")
print("Tablas unidas")

# Paso 6: Transformaciones
# Estandarizar fechas si existe columna DATE
if "DATE" in df_merge.columns:
    df_merge["DATE"] = pd.to_datetime(df_merge["DATE"], errors="coerce")

# Crear columna calculada de ventas totales si existen QUANTITY y PRICE
if {"QUANTITY","PRICE"} <= set(df_merge.columns):
    df_merge["TOTAL_SALES"] = df_merge["QUANTITY"] * df_merge["PRICE"]

print("Transformaciones aplicadas")

# Paso 7: Guardar archivo final
df_merge.to_excel("tarea_final.xlsx", index=False)
df_merge.to_csv("tarea_final.csv", index=False)
print("Archivos guardados correctamente")

# Revisión rápida del resultado
print(df_merge.head())
print(df_merge.shape)